# Tutorial 1. Classification, tokenization and generation

In [1]:
# %pip install -q "datasets<5" "transformers>=4.48,<5" "accelerate<2" "mauve-text==0.4.0"

In [1]:
import os
import re
import gc
import time
import copy
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tokenizers import Tokenizer, models, trainers, normalizers, pre_tokenizers, decoders
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"
SEED = 42
CLASS_VOCAB_SIZE = 4096
STORY_VOCAB_SIZE = 4096
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_TRAIN_SIZE = 60000 if DEVICE.type == "cuda" else 12000
CLASS_VALID_SIZE = 6000 if DEVICE.type == "cuda" else 2000
CLASS_MAX_LEN = 96
CLASS_EPOCHS = 4
CLASS_BATCH_SIZE = 256 if DEVICE.type == "cuda" else 64
LM_TRAIN_SIZE = 50000 if DEVICE.type == "cuda" else 6000
LM_VALID_SIZE = 2000 if DEVICE.type == "cuda" else 500
LM_MAX_LEN = 160 if DEVICE.type == "cuda" else 128
LM_EPOCHS = 3 if DEVICE.type == "cuda" else 2
LM_BATCH_SIZE = 128 if DEVICE.type == "cuda" else 32
GENERATION_SIZE = 192 if DEVICE.type == "cuda" else 48
GENERATION_TOKENS = 64 if DEVICE.type == "cuda" else 48
JUDGE_SIZE = 16 if DEVICE.type == "cuda" else 8
PAIR_SIZE = 12 if DEVICE.type == "cuda" else 6
JUDGE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct" if DEVICE.type == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("high")
DEVICE

device(type='cpu')

## 1. TF–IDF

**TF–IDF**
$$
\mathrm{tf}(t, d) = \frac{1}{N_d}\sum_{i = 1}^{N_d} [t = d_i]; \qquad \mathrm{idf}(t, D) = \log \frac{|D|}{\sum_{d \in D} [t \in d]}
$$

$$
h(d) = \{\mathrm{tf}\text{-}\mathrm{idf}(t, d, D)\}_{t \in Vocab}
$$

**Logistic regression**

$$
z = W \cdot h(d) + b,
\qquad
p_\theta(y=k\mid d)=\frac{e^{z_k}}{\sum_{j=1}^{C}e^{z_j}},
\qquad
\mathcal L=-\frac1N\sum_{i=1}^{N}\log p_\theta(y_i\mid d_i).
$$

In [2]:
dataset = load_dataset("ag_news")
full_train = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas().reset_index(drop=True)
train_df, valid_df = train_test_split(
    full_train,
    train_size=CLASS_TRAIN_SIZE,
    test_size=CLASS_VALID_SIZE,
    stratify=full_train["label"],
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

train_texts = train_df["text"].tolist()
valid_texts = valid_df["text"].tolist()
test_texts = test_df["text"].tolist()
y_train = train_df["label"].to_numpy()
y_valid = valid_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()
label_names = ["World", "Sports", "Business", "Sci/Tech"]

pd.DataFrame({
    "split": ["train", "valid", "test"],
    "size": [len(train_df), len(valid_df), len(test_df)],
    "majority_accuracy": [
        train_df["label"].value_counts(normalize=True).max(),
        valid_df["label"].value_counts(normalize=True).max(),
        test_df["label"].value_counts(normalize=True).max(),
    ],
})

,split,size,majority_accuracy
0,train,12000,0.25
1,valid,2000,0.25
2,test,7600,0.25


In [19]:
list(train_df['text'][:2])

['Impact May Be Small As  Fannie, Freddie Regroup Mortgage securities traders and strategists say if Fannie Mae does reduce the number of mortgages it buys, it could lead to a small, short-term increase in mortgage interest rates paid by consumers, probably no more than 0.1 percent.',
 "Warner Beats Out Manning As Giants Starter (AP) AP - Eli Manning remains the New York Giants' quarterback of the future. For now, the job belongs to Kurt Warner."]

In [3]:
tfidf_models = {}
tfidf_rows = []

for name, ngram_range, max_features in [
    ("word TF-IDF 4k", (1, 1), CLASS_VOCAB_SIZE),
    ("word TF-IDF 50k", (1, 1), 50000),
    ("word 1–2-gram TF-IDF 50k", (1, 2), 50000),
]:
    started = time.perf_counter()
    vectorizer = TfidfVectorizer(
        ngram_range=ngram_range,
        min_df=2,
        max_features=max_features,
    )
    x_train = vectorizer.fit_transform(train_texts)
    x_valid = vectorizer.transform(valid_texts)
    x_test = vectorizer.transform(test_texts)
    model = LogisticRegression(C=4.0, max_iter=400, tol=1e-3)
    model.fit(x_train, y_train)
    valid_pred = model.predict(x_valid)
    test_pred = model.predict(x_test)
    tfidf_models[name] = (vectorizer, model)
    tfidf_rows.append({
        "model": name + " + LR",
        "vocab": x_train.shape[1],
        "parameters": model.coef_.size + model.intercept_.size,
        "matrix_density": x_train.nnz / (x_train.shape[0] * x_train.shape[1]),
        "valid_accuracy": accuracy_score(y_valid, valid_pred),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_macro_f1": f1_score(y_test, test_pred, average="macro"),
        "train_seconds": time.perf_counter() - started,
    })

tfidf_results = pd.DataFrame(tfidf_rows)
tfidf_results

,model,vocab,parameters,matrix_density,valid_accuracy,test_accuracy,test_macro_f1,train_seconds
0,word TF-IDF 8k + LR,4096,16388,0.006422,0.8765,0.883289,0.882978,1.05061
1,word TF-IDF 50k + LR,15267,61072,0.001985,0.8880,0.891842,0.891563,1.68188
2,word 1–2-gram TF-IDF 50k + LR,50000,200004,0.000969,0.8890,0.893553,0.893205,4.36628


A linear model, unlike neural networks, is very easy to interpret. For the class $k$

$$
z_k=b_k+\sum_{w\in V}W_{kw}h_w,
\qquad
\operatorname{contribution}_{k,w}=W_{kw}h_w.
$$

A positive $W_{kw}$ means, that the presence of a feature moves the logit toward the class $k$.

In [4]:
best_tfidf_name = tfidf_results.sort_values("valid_accuracy").iloc[-1]["model"].replace(" + LR", "")
best_vectorizer, best_lr = tfidf_models[best_tfidf_name]
feature_names = np.array(best_vectorizer.get_feature_names_out())
top_ids = np.argsort(best_lr.coef_, axis=1)[:, -12:][:, ::-1]
pd.DataFrame({label_names[i]: feature_names[top_ids[i]] for i in range(4)})

,World,Sports,Business,Sci/Tech
0,iraq,team,tax,nasa
1,afp,coach,oil,microsoft
2,president,cup,bank,scientists
3,nuclear,league,company,software
4,iran,football,prices,internet
5,military,olympic,economy,computer
6,israel,baseball,billion,space
7,minister,sports,sales,online
8,leader,athens,us,web
9,iraqi,39,insurance,linux


## 2. Vocabulary size problem

* A **small** vocabulary won't fit all words
* A **large** vocabulary will contain too much rare words, expanding the model size and leading to overfitting.

For neural networks token embeddings weigh **A LOT**. 

$$
\begin{aligned}
\#\theta_{\text{LR}} &= CV + C,\\
\#\theta_{\text{Emb}} &= Vd,\\
\#\theta_{\text{RNN}} &= h(d+h+2)+C(h+1),\\
\end{aligned}
$$

Here $d$ is the embedding dimension, $h$ is the hidden size, $C$ is the number of classes.

In [5]:
V = CLASS_VOCAB_SIZE
d = 128
h = 256
C = 4
pd.DataFrame([
    {
        "model": "TF-IDF + LR",
        "input / embedding": 0,
        "recurrent block": 0,
        "output head": C * V + C,
        "total": C * V + C,
    },
    {
        "model": "RNN classifier",
        "input / embedding": V * d,
        "recurrent block": h * (d + h + 2),
        "output head": C * h + C,
        "total": V * d + h * (d + h + 2) + C * h + C,
    },
])

,model,input / embedding,recurrent block,output head,total
0,TF-IDF + LR,0,0,16388,16388
1,RNN classifier,524288,98816,1028,624132


#### Vocabulary size reduction

* Stemming
* Lemmatization
* Replace rare words with `[UNK]`.

This is a working vocabulary compression, but it results in information loss.

## 3. Tokenization: BPE and WordPiece

Let's build an optimal vocabulary of fixed size that includes only the most important tokens.

### Byte-Pair Encoding
**Idea:** Start from a symbolic vocabulary. Merge the most common pairs of tokens into one.


**Corpus:** `("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)`

Split it in symbols.   
`("h" "u" "g", 10), ("p" "u" "g", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "u" "g" "s", 5)`

**Vocabulary:** `["b", "g", "h", "n", "p", "s", "u", "ug"]`   
**Corpus:** `("h" "ug", 10), ("p" "ug", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "ug" "s", 5)`

**Vocabulary:** `["b", "g", "h", "n", "p", "s", "u", "ug", "un"]`   
**Corpus:** `("h" "ug", 10), ("p" "ug", 5), ("p" "un", 12), ("b" "un", 4), ("h" "ug" "s", 5)`

**Vocabulary:** `["b", "g", "h", "n", "p", "s", "u", "ug", "un", "hug"]`   
**Corpus:** `("hug", 10), ("p" "ug", 5), ("p" "un", 12), ("b" "un", 4), ("hug" "s", 5)`


**BPE** combines the most frequent adjacent pair

$$
\operatorname{BPE}_{\operatorname{score}}(a, b) = c(a, b) \qquad (a^*,b^*)=\arg\max_{a,b} \operatorname{BPE}_{\operatorname{score}} (a,b),
\qquad
V\leftarrow V\cup\{a^*b^*\}.
$$

**WordPiece** prefers pairs that occur together frequently relative to individual frequencies:

$$
\operatorname{WordPiece}_{\operatorname{score}}(a,b)=\frac{c(a,b)}{c(a)c(b)}.
$$

In [6]:
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
WORD_PATTERN = re.compile(r"\w+(?:'\w+)?|[^\w\s]", re.UNICODE)

word_counts = Counter()
for text in train_texts:
    word_counts.update(WORD_PATTERN.findall(text.lower()))
word_vocab = SPECIAL_TOKENS + [w for w, _ in word_counts.most_common(CLASS_VOCAB_SIZE - len(SPECIAL_TOKENS))]
word2id = {w: i for i, w in enumerate(word_vocab)}

bpe_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
bpe_tokenizer.normalizer = normalizers.Lowercase()
bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_tokenizer.decoder = decoders.ByteLevel()
bpe_trainer = trainers.BpeTrainer(
    vocab_size=CLASS_VOCAB_SIZE,
    min_frequency=2,
    special_tokens=SPECIAL_TOKENS,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)
bpe_tokenizer.train_from_iterator(train_texts, trainer=bpe_trainer, length=len(train_texts))

wordpiece_tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))
wordpiece_tokenizer.normalizer = normalizers.Sequence([normalizers.NFKC(), normalizers.Lowercase()])
wordpiece_tokenizer.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
wordpiece_tokenizer.decoder = decoders.WordPiece(prefix="##")
wordpiece_trainer = trainers.WordPieceTrainer(
    vocab_size=CLASS_VOCAB_SIZE,
    min_frequency=2,
    special_tokens=SPECIAL_TOKENS,
)
wordpiece_tokenizer.train_from_iterator(train_texts, trainer=wordpiece_trainer, length=len(train_texts))

assert word2id["[PAD]"] == bpe_tokenizer.token_to_id("[PAD]") == 0

In [25]:
examples = [
    "Microsoft launches a new AI platform for developers.",
    "The electroencephalographically-inspired model was retrained.",
    "This sem is lowkey goated twin fr fr",
    "GPU-accelerated NLP costs $1.99.",
    "Unbelievablly fast classificatoin",
    "I miss my ex so much, please come back",
]

rows = []
for text in examples:
    word_tokens = WORD_PATTERN.findall(text.lower())
    word_tokens = [t if t in word2id else "[UNK]" for t in word_tokens]
    rows.append({
        "text": text,
        "word": word_tokens,
        "BPE": [t for t in bpe_tokenizer.encode(text).tokens],
        "WordPiece": wordpiece_tokenizer.encode(text).tokens,
    })

pd.DataFrame(rows)

,text,word,BPE,WordPiece
0,Microsoft launches a new AI platform for devel...,"[microsoft, launches, a, new, [UNK], platform,...","[microsoft, Ġlaunches, Ġa, Ġnew, Ġa, i, Ġplatf...","[microsoft, launches, a, new, a, ##i, platform..."
1,The electroencephalographically-inspired model...,"[the, [UNK], -, [UNK], model, was, [UNK], .]","[the, Ġelect, ro, ence, ph, al, ograph, ically...","[the, elect, ##ro, ##ence, ##ph, ##al, ##ograp..."
2,This sem is lowkey goated twin fr fr,"[this, [UNK], is, [UNK], [UNK], twin, [UNK], [...","[th, is, Ġsem, Ġis, Ġlow, key, Ġgo, ated, Ġtw,...","[this, sem, is, low, ##key, go, ##ated, tw, ##..."
3,GPU-accelerated NLP costs $1.99.,"[[UNK], -, [UNK], [UNK], costs, $, 1, ., 99, .]","[g, p, u, -, ac, cel, er, ated, Ġn, l, p, Ġcos...","[g, ##p, ##u, -, acc, ##el, ##erate, ##d, nl, ..."
4,Unbelievablly fast classificatoin,"[[UNK], fast, [UNK]]","[un, b, el, ie, v, ab, l, ly, Ġfast, Ġclass, i...","[un, ##be, ##l, ##ie, ##v, ##ab, ##ll, ##y, fa..."
5,"I miss my ex so much, please come back","[i, miss, my, ex, so, much, ,, [UNK], come, back]","[i, Ġmiss, Ġmy, Ġex, Ġso, Ġmuch, ,, Ġple, ase,...","[i, miss, my, ex, so, much, ,, ple, ##ase, com..."


In [8]:
stat_texts = test_texts[:2000]
word_lengths = []
word_unk = 0
word_total = 0
bpe_lengths = []
bpe_unk = 0
bpe_total = 0
wp_lengths = []
wp_unk = 0
wp_total = 0

for text in stat_texts:
    wt = WORD_PATTERN.findall(text.lower())
    word_lengths.append(len(wt))
    word_unk += sum(t not in word2id for t in wt)
    word_total += len(wt)

for encoding in bpe_tokenizer.encode_batch(stat_texts):
    bpe_lengths.append(len(encoding.ids))
    bpe_unk += encoding.ids.count(bpe_tokenizer.token_to_id("[UNK]"))
    bpe_total += len(encoding.ids)

for encoding in wordpiece_tokenizer.encode_batch(stat_texts):
    wp_lengths.append(len(encoding.ids))
    wp_unk += encoding.ids.count(wordpiece_tokenizer.token_to_id("[UNK]"))
    wp_total += len(encoding.ids)

pd.DataFrame([
    {"tokenizer": "word", "vocab": len(word_vocab), "mean_length": np.mean(word_lengths), "p95_length": np.quantile(word_lengths, 0.95), "unk_rate": word_unk / word_total},
    {"tokenizer": "byte BPE", "vocab": bpe_tokenizer.get_vocab_size(), "mean_length": np.mean(bpe_lengths), "p95_length": np.quantile(bpe_lengths, 0.95), "unk_rate": bpe_unk / bpe_total},
    {"tokenizer": "WordPiece", "vocab": wordpiece_tokenizer.get_vocab_size(), "mean_length": np.mean(wp_lengths), "p95_length": np.quantile(wp_lengths, 0.95), "unk_rate": wp_unk / wp_total},
])

,tokenizer,vocab,mean_length,p95_length,unk_rate
0,word,4096,46.8980,73.0,0.133641
1,byte BPE,4096,64.8160,99.0,0.000000
2,WordPiece,4096,62.7075,98.0,0.000000


## 4. word-RNN vs BPE-RNN

RNN reads tokens in left-to-right order.
$$
e_t=E[x_t],
\qquad
h_t=\tanh(W_e e_t+U h_{t-1}+b),
$$

For classification we use the last state of the real text:

$$
p_\theta(y\mid x)=\operatorname{softmax}(Ah_T+a).
$$

Word-RNN and BPE-RNN have the same $V$, $d$, $h$, number of epochs, and $T_{\max}=96$ limit. Only the tokenization changes. For a single-layer implementation

$$
\#\theta=Vd+h(d+h+2)+C(h+1).
$$

In [9]:
PAD_ID = 0
UNK_ID = 1


def encode_word_matrix(texts, max_length):
    matrix = np.full((len(texts), max_length), PAD_ID, dtype=np.int64)
    lengths = np.zeros(len(texts), dtype=np.int64)
    oov_rates = np.zeros(len(texts), dtype=np.float32)
    for i, text in enumerate(tqdm(texts)):
        tokens = WORD_PATTERN.findall(text.lower())
        ids = [word2id.get(token, UNK_ID) for token in tokens][:max_length]
        lengths[i] = max(1, len(ids))
        oov_rates[i] = np.mean([token not in word2id for token in tokens]) if tokens else 0.0
        matrix[i, :len(ids)] = ids
    return torch.from_numpy(matrix), torch.from_numpy(lengths), oov_rates


def encode_bpe_matrix(texts, max_length):
    encodings = bpe_tokenizer.encode_batch(texts)
    matrix = np.full((len(texts), max_length), PAD_ID, dtype=np.int64)
    lengths = np.zeros(len(texts), dtype=np.int64)
    for i, encoding in enumerate(encodings):
        ids = encoding.ids[:max_length]
        lengths[i] = max(1, len(ids))
        matrix[i, :len(ids)] = ids
    return torch.from_numpy(matrix), torch.from_numpy(lengths)


word_x_train, word_l_train, _ = encode_word_matrix(train_texts, CLASS_MAX_LEN)
word_x_valid, word_l_valid, _ = encode_word_matrix(valid_texts, CLASS_MAX_LEN)
word_x_test, word_l_test, word_test_oov = encode_word_matrix(test_texts, CLASS_MAX_LEN)

bpe_x_train, bpe_l_train = encode_bpe_matrix(train_texts, CLASS_MAX_LEN)
bpe_x_valid, bpe_l_valid = encode_bpe_matrix(valid_texts, CLASS_MAX_LEN)
bpe_x_test, bpe_l_test = encode_bpe_matrix(test_texts, CLASS_MAX_LEN)

  0%|          | 0/12000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/7600 [00:00<?, ?it/s]

In [10]:
def make_loader(x, lengths, labels, shuffle, batch_size):
    return DataLoader(
        TensorDataset(x, lengths, torch.as_tensor(labels, dtype=torch.long)),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda",
    )

word_train_loader = make_loader(word_x_train, word_l_train, y_train, True, CLASS_BATCH_SIZE)
word_valid_loader = make_loader(word_x_valid, word_l_valid, y_valid, False, CLASS_BATCH_SIZE)
word_test_loader = make_loader(word_x_test, word_l_test, y_test, False, CLASS_BATCH_SIZE)

bpe_train_loader = make_loader(bpe_x_train, bpe_l_train, y_train, True, CLASS_BATCH_SIZE)
bpe_valid_loader = make_loader(bpe_x_valid, bpe_l_valid, y_valid, False, CLASS_BATCH_SIZE)
bpe_test_loader = make_loader(bpe_x_test, bpe_l_test, y_test, False, CLASS_BATCH_SIZE)

In [11]:
class TextRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_size=256, classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity="tanh")
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Linear(hidden_size, classes)

    def forward(self, input_ids, lengths):
        hidden_states, _ = self.rnn(self.embedding(input_ids))
        batch_ids = torch.arange(len(input_ids), device=input_ids.device)
        last_hidden = hidden_states[batch_ids, lengths - 1]
        return self.head(self.dropout(last_hidden))

In [12]:
@torch.inference_mode()
def evaluate_classifier(model, loader):
    model.eval()
    losses = []
    predictions = []
    targets = []
    for input_ids, lengths, labels in loader:
        input_ids = input_ids.to(DEVICE, non_blocking=True)
        lengths = lengths.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(input_ids, lengths)
        losses.append(F.cross_entropy(logits, labels).item() * len(labels))
        predictions.extend(logits.argmax(dim=-1).cpu().tolist())
        targets.extend(labels.cpu().tolist())
    return sum(losses) / len(targets), accuracy_score(targets, predictions), np.array(predictions)


def fit_classifier(train_loader, valid_loader, test_loader, vocab_size):
    torch.manual_seed(SEED)
    model = TextRNN(vocab_size).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
    best_accuracy = -1.0
    best_state = None
    history = []
    started = time.perf_counter()

    for epoch in range(CLASS_EPOCHS):
        model.train()
        total_loss = 0.0
        total_size = 0
        grad_norms = []
        for input_ids, lengths, labels in tqdm(train_loader, leave=False):
            input_ids = input_ids.to(DEVICE, non_blocking=True)
            lengths = lengths.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids, lengths)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            grad_norms.append(float(torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)))
            optimizer.step()
            total_loss += loss.item() * len(labels)
            total_size += len(labels)

        valid_loss, valid_accuracy, _ = evaluate_classifier(model, valid_loader)
        history.append({
            "epoch": epoch + 1,
            "train_loss": total_loss / total_size,
            "valid_loss": valid_loss,
            "valid_accuracy": valid_accuracy,
            "mean_grad_norm_before_clip": np.mean(grad_norms),
        })
        if valid_accuracy > best_accuracy:
            best_accuracy = valid_accuracy
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    test_loss, test_accuracy, test_predictions = evaluate_classifier(model, test_loader)
    return model, pd.DataFrame(history), test_loss, test_accuracy, test_predictions, time.perf_counter() - started

In [13]:
word_rnn, word_history, word_test_loss, word_test_accuracy, word_test_pred, word_time = fit_classifier(
    word_train_loader,
    word_valid_loader,
    word_test_loader,
    len(word_vocab),
)
word_history

  0%|          | 0/188 [00:00<?, ?it/s]

  0%|          | 0/188 [00:00<?, ?it/s]

  0%|          | 0/188 [00:00<?, ?it/s]

  0%|          | 0/188 [00:00<?, ?it/s]

,epoch,train_loss,valid_loss,valid_accuracy,mean_grad_norm_before_clip
0,1,1.344101,1.345886,0.3485,2.919090
1,2,1.112472,1.010678,0.5320,2.189726
2,3,0.951251,1.077802,0.5400,2.441441
3,4,0.987976,1.476176,0.4100,3.934153


In [14]:
bpe_rnn, bpe_history, bpe_test_loss, bpe_test_accuracy, bpe_test_pred, bpe_time = fit_classifier(
    bpe_train_loader,
    bpe_valid_loader,
    bpe_test_loader,
    bpe_tokenizer.get_vocab_size(),
)
bpe_history

  0%|          | 0/188 [00:00<?, ?it/s]

  0%|          | 0/188 [00:00<?, ?it/s]

  0%|          | 0/188 [00:00<?, ?it/s]

  0%|          | 0/188 [00:00<?, ?it/s]

,epoch,train_loss,valid_loss,valid_accuracy,mean_grad_norm_before_clip
0,1,1.328231,1.220275,0.4260,3.144252
1,2,1.188322,1.151888,0.4650,3.463077
2,3,1.024907,1.082009,0.5125,2.226224
3,4,0.906940,1.007282,0.5765,2.976319


In [15]:
best_tfidf_row = tfidf_results.sort_values("valid_accuracy").iloc[-1]
capped_tfidf_row = tfidf_results[tfidf_results["model"].str.contains("8k")].iloc[0]
classification_results = pd.DataFrame([
    {
        "model": capped_tfidf_row["model"],
        "vocab": int(capped_tfidf_row["vocab"]),
        "parameters": int(capped_tfidf_row["parameters"]),
        "mean_tokens": np.mean(word_lengths),
        "test_accuracy": capped_tfidf_row["test_accuracy"],
        "test_macro_f1": capped_tfidf_row["test_macro_f1"],
        "train_seconds": capped_tfidf_row["train_seconds"],
    },
    {
        "model": best_tfidf_row["model"],
        "vocab": int(best_tfidf_row["vocab"]),
        "parameters": int(best_tfidf_row["parameters"]),
        "mean_tokens": np.mean(word_lengths),
        "test_accuracy": best_tfidf_row["test_accuracy"],
        "test_macro_f1": best_tfidf_row["test_macro_f1"],
        "train_seconds": best_tfidf_row["train_seconds"],
    },
    {
        "model": "word RNN",
        "vocab": len(word_vocab),
        "parameters": sum(p.numel() for p in word_rnn.parameters()),
        "mean_tokens": np.mean(word_lengths),
        "test_accuracy": word_test_accuracy,
        "test_macro_f1": f1_score(y_test, word_test_pred, average="macro"),
        "train_seconds": word_time,
    },
    {
        "model": "BPE RNN",
        "vocab": bpe_tokenizer.get_vocab_size(),
        "parameters": sum(p.numel() for p in bpe_rnn.parameters()),
        "mean_tokens": np.mean(bpe_lengths),
        "test_accuracy": bpe_test_accuracy,
        "test_macro_f1": f1_score(y_test, bpe_test_pred, average="macro"),
        "train_seconds": bpe_time,
    },
])
classification_results

,model,vocab,parameters,mean_tokens,test_accuracy,test_macro_f1,train_seconds
0,word TF-IDF 8k + LR,4096,16388,46.898,0.883289,0.882978,1.050610
1,word 1–2-gram TF-IDF 50k + LR,50000,200004,46.898,0.893553,0.893205,4.366280
2,word RNN,4096,624132,46.898,0.540526,0.526665,34.401845
3,BPE RNN,4096,624132,64.816,0.552500,0.544693,37.738957


## 5. Generation

In classification, one sequence yields one of the $C$ classes. In a language model, each step is a classification over the entire vocabulary:

$$
\text{classification: } h_T\mapsto\mathbb R^C,
\qquad
\text{generation: } h_t\mapsto\mathbb R^V\quad\forall t.
$$

$$
p_\theta(t_{1:T})=\prod_{i=1}^{T}p_\theta(t_i\mid t_{<i}),
\qquad
z_i=Oh_i+c\in\mathbb R^V.
$$

Therefore, tokenization defines both the input and the space of possible outputs. For generation, we'll use TinyStories.

In [27]:
story_train_stream = load_dataset("skeskinen/TinyStories-hf", split="train", streaming=True).shuffle(seed=SEED, buffer_size=20000)
story_valid_stream = load_dataset("skeskinen/TinyStories-hf", split="validation", streaming=True).shuffle(seed=SEED, buffer_size=5000)

lm_train_texts = []
for row in story_train_stream:
    text = row["text"].strip()
    if len(text.split()) >= 40 and "�" not in text and "â" not in text:
        lm_train_texts.append(text)
    if len(lm_train_texts) == LM_TRAIN_SIZE:
        break

lm_valid_texts = []
for row in story_valid_stream:
    text = row["text"].strip()
    if len(text.split()) >= 40 and "�" not in text and "â" not in text:
        lm_valid_texts.append(text)
    if len(lm_valid_texts) == LM_VALID_SIZE:
        break

story_word_counts = Counter()
for text in lm_train_texts:
    story_word_counts.update(WORD_PATTERN.findall(text.lower()))
story_word_vocab = SPECIAL_TOKENS + [w for w, _ in story_word_counts.most_common(STORY_VOCAB_SIZE - len(SPECIAL_TOKENS))]
story_word2id = {w: i for i, w in enumerate(story_word_vocab)}

story_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
story_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
story_tokenizer.decoder = decoders.ByteLevel()
story_trainer = trainers.BpeTrainer(
    vocab_size=STORY_VOCAB_SIZE,
    min_frequency=2,
    special_tokens=SPECIAL_TOKENS,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)
story_tokenizer.train_from_iterator(lm_train_texts, trainer=story_trainer, length=len(lm_train_texts))

pd.DataFrame({
    "split": ["train", "validation"],
    "stories": [len(lm_train_texts), len(lm_valid_texts)],
    "mean_words": [np.mean([len(WORD_PATTERN.findall(x)) for x in lm_train_texts]), np.mean([len(WORD_PATTERN.findall(x)) for x in lm_valid_texts])],
})

,split,stories,mean_words
0,train,6000,217.686167
1,validation,500,187.118000


In [28]:
story_example = lm_valid_texts[0]
story_encoding = story_tokenizer.encode(story_example)
pd.DataFrame([{
    "text": story_example[:300] + "...",
    "word tokens": WORD_PATTERN.findall(story_example.lower())[:30],
    "BPE tokens": [t.replace("Ġ", "▁") for t in story_encoding.tokens[:40]],
}])

,text,word tokens,BPE tokens
0,Once upon a time there was a girl named Sally....,"[once, upon, a, time, there, was, a, girl, nam...","[Once, ▁upon, ▁a, ▁time, ▁there, ▁was, ▁a, ▁gi..."


Let's compare tokenizers using unigram models:

$$
p(t_{1:T})=\prod_{i=1}^{T}p(t_i),
\qquad
\widehat p(v)=\frac{c(v)+\alpha}{\sum_{u\in V}c(u)+\alpha V}.
$$

For tokenizer $\tau$

$$
\operatorname{NLL}_\tau(x)
=-\sum_{i=1}^{T_\tau(x)}\log p(t_i),
\qquad
\operatorname{PPL}_{\tau}
=\exp\left(\frac{\operatorname{NLL}_\tau(x)}{T_\tau(x)}\right).
$$

In [29]:
def unigram_metrics(train_sequences, valid_sequences, vocab_size, valid_bytes, alpha=0.1):
    counts = np.full(vocab_size, alpha, dtype=np.float64)
    for sequence in train_sequences:
        np.add.at(counts, sequence, 1)
    probabilities = counts / counts.sum()
    total_nll = 0.0
    total_tokens = 0
    for sequence in valid_sequences:
        total_nll -= np.log(probabilities[sequence]).sum()
        total_tokens += len(sequence)
    nll = total_nll / total_tokens
    return nll, np.exp(nll), total_nll / (valid_bytes * np.log(2)), total_tokens / len(valid_sequences)

unigram_train = lm_train_texts[:10000]
unigram_valid = lm_valid_texts[:1000]
valid_bytes = sum(len(text.encode("utf-8")) for text in unigram_valid)

byte_train = [np.frombuffer(text.encode("utf-8"), dtype=np.uint8) for text in unigram_train]
byte_valid = [np.frombuffer(text.encode("utf-8"), dtype=np.uint8) for text in unigram_valid]
word_train = [np.array([story_word2id.get(t, 1) for t in WORD_PATTERN.findall(text.lower())], dtype=np.int64) for text in unigram_train]
word_valid = [np.array([story_word2id.get(t, 1) for t in WORD_PATTERN.findall(text.lower())], dtype=np.int64) for text in unigram_valid]
bpe_train = [np.array(e.ids, dtype=np.int64) for e in story_tokenizer.encode_batch(unigram_train)]
bpe_valid = [np.array(e.ids, dtype=np.int64) for e in story_tokenizer.encode_batch(unigram_valid)]

unigram_rows = []
for name, train_sequences, valid_sequences, vocab_size, lossless in [
    ("byte", byte_train, byte_valid, 256, True),
    ("word", word_train, word_valid, len(story_word_vocab), False),
    ("BPE", bpe_train, bpe_valid, story_tokenizer.get_vocab_size(), True),
]:
    nll, ppl, bpb, mean_length = unigram_metrics(train_sequences, valid_sequences, vocab_size, valid_bytes)
    unigram_rows.append({
        "tokenizer": name,
        "vocab": vocab_size,
        "mean_tokens": mean_length,
        "NLL/token": nll,
        "PPL/token": ppl,
        "bits/byte": bpb if lossless else np.nan,
    })

pd.DataFrame(unigram_rows)

,tokenizer,vocab,mean_tokens,NLL/token,PPL/token,bits/byte
0,byte,256,813.270,3.058345,21.292299,4.412260
1,word,4096,187.118,5.654032,285.439931,NaN
2,BPE,4096,204.826,6.027043,414.487478,2.189925


For lossless tokenization, one can normalize the negative log-likelihood to the same number of bytes in the original string:

$$
\operatorname{BPB}(x)
=\frac{\operatorname{NLL}(x)}{B(x)\ln 2},
\qquad
\operatorname{PPL}_{\text{byte-equiv}}=2^{\operatorname{BPB}}.
$$

If the word-tokenizer has replaced part of the string with `[UNK]`, it no longer sets the probability of the original string, so we do not calculate BPB for it here.

## 6. LSTM

ванильная RNN хранит всё прошлое в одном $h_t$. Для нашей `tanh`-RNN градиент между далёкими состояниями содержит произведение якобианов:

$$
\frac{\partial h_t}{\partial h_k}
=
\prod_{j=k+1}^{t}D_jU,
\qquad
D_j=\operatorname{diag}(1-h_j^2).
$$

Если нормы множителей систематически меньше или больше единицы, градиент затухает или взрывается. LSTM добавляет отдельную память $c_t$ и управляемый путь её обновления:

$$
\begin{aligned}
f_t&=\sigma(W_f e_t+U_fh_{t-1}+b_f),\\
i_t&=\sigma(W_i e_t+U_ih_{t-1}+b_i),\\
\widetilde c_t&=\tanh(W_c e_t+U_ch_{t-1}+b_c),\\
c_t&=f_t\odot c_{t-1}+i_t\odot\widetilde c_t,\\
o_t&=\sigma(W_o e_t+U_oh_{t-1}+b_o),\\
h_t&=o_t\odot\tanh(c_t).
\end{aligned}
$$

По прямому пути памяти локальный якобиан имеет вид

$$
\left.\frac{\partial c_t}{\partial c_{t-1}}\right|_{\text{memory path}}=\operatorname{diag}(f_t),
$$

поэтому модель может сама выбирать, сколько памяти и градиента пропустить дальше.

Обучаем предсказывать каждый следующий BPE-токен:

$$
\mathcal L(\theta)
=-\frac{1}{\sum_{m,t}M_{mt}}
\sum_{m,t}M_{mt}\log p_\theta(t_{m,t+1}\mid t_{m,\le t}).
$$

Чтобы избежать взрыва градиента клипаем градиент:

$$
g\leftarrow g\min\left(1,\frac{1}{\|g\|_2}\right).
$$

In [30]:
STORY_PAD = story_tokenizer.token_to_id("[PAD]")
STORY_UNK = story_tokenizer.token_to_id("[UNK]")
STORY_BOS = story_tokenizer.token_to_id("[BOS]")
STORY_EOS = story_tokenizer.token_to_id("[EOS]")
STORY_VOCAB = story_tokenizer.get_vocab_size()


def encode_lm_matrix(texts, max_length):
    encodings = story_tokenizer.encode_batch(texts)
    matrix = np.full((len(texts), max_length), STORY_PAD, dtype=np.int64)
    for i, encoding in enumerate(encodings):
        ids = [STORY_BOS] + encoding.ids
        if len(ids) < max_length:
            ids.append(STORY_EOS)
        ids = ids[:max_length]
        matrix[i, :len(ids)] = ids
    return torch.from_numpy(matrix)

lm_train_ids = encode_lm_matrix(lm_train_texts, LM_MAX_LEN)
lm_valid_ids = encode_lm_matrix(lm_valid_texts, LM_MAX_LEN)
lm_train_loader = DataLoader(TensorDataset(lm_train_ids), batch_size=LM_BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=DEVICE.type == "cuda")
lm_valid_loader = DataLoader(TensorDataset(lm_valid_ids), batch_size=LM_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=DEVICE.type == "cuda")

In [31]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_size=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=STORY_PAD)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids, state=None):
        hidden_states, state = self.lstm(self.embedding(input_ids), state)
        return self.head(hidden_states), state


lm = LSTMLM(STORY_VOCAB).to(DEVICE)
pd.DataFrame([
    {"block": "embedding", "parameters": lm.embedding.weight.numel()},
    {"block": "LSTM", "parameters": sum(p.numel() for p in lm.lstm.parameters())},
    {"block": "output head", "parameters": sum(p.numel() for p in lm.head.parameters())},
])

,block,parameters
0,embedding,524288
1,LSTM,395264
2,output head,1052672


In [32]:
@torch.inference_mode()
def evaluate_lm(model, loader):
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    for (input_ids,) in loader:
        input_ids = input_ids.to(DEVICE, non_blocking=True)
        logits, _ = model(input_ids[:, :-1])
        targets = input_ids[:, 1:]
        total_nll += F.cross_entropy(logits.reshape(-1, STORY_VOCAB), targets.reshape(-1), ignore_index=STORY_PAD, reduction="sum").item()
        total_tokens += (targets != STORY_PAD).sum().item()
    return total_nll / total_tokens


optimizer = torch.optim.AdamW(lm.parameters(), lr=2e-3, weight_decay=1e-4)
best_nll = float("inf")
best_state = None
lm_history = []

for epoch in range(LM_EPOCHS):
    lm.train()
    total_nll = 0.0
    total_tokens = 0
    grad_norms = []
    for (input_ids,) in tqdm(lm_train_loader, leave=False):
        input_ids = input_ids.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits, _ = lm(input_ids[:, :-1])
        targets = input_ids[:, 1:]
        loss_sum = F.cross_entropy(logits.reshape(-1, STORY_VOCAB), targets.reshape(-1), ignore_index=STORY_PAD, reduction="sum")
        token_count = (targets != STORY_PAD).sum()
        loss = loss_sum / token_count
        loss.backward()
        grad_norms.append(float(torch.nn.utils.clip_grad_norm_(lm.parameters(), 1.0)))
        optimizer.step()
        total_nll += loss_sum.item()
        total_tokens += token_count.item()

    train_nll = total_nll / total_tokens
    valid_nll = evaluate_lm(lm, lm_valid_loader)
    lm_history.append({
        "epoch": epoch + 1,
        "train_NLL": train_nll,
        "valid_NLL": valid_nll,
        "valid_PPL": np.exp(valid_nll),
        "mean_grad_norm_before_clip": np.mean(grad_norms),
    })
    if valid_nll < best_nll:
        best_nll = valid_nll
        best_state = copy.deepcopy(lm.state_dict())

lm.load_state_dict(best_state)
pd.DataFrame(lm_history)

  0%|          | 0/188 [00:00<?, ?it/s]

  0%|          | 0/188 [00:00<?, ?it/s]

,epoch,train_NLL,valid_NLL,valid_PPL,mean_grad_norm_before_clip
0,1,4.816904,3.955422,52.217725,0.281208
1,2,3.683684,3.535937,34.327170,0.218114


Температура меняет распределение сэмплирования:

$$
p_{\theta,\tau}(v\mid h)
=\frac{\exp(z_v/\tau)}{\sum_{u\in V}\exp(z_u/\tau)}.
$$

In [33]:
encoded_story_valid = story_tokenizer.encode_batch(lm_valid_texts)
prefix_length = 16
selected = [i for i, e in enumerate(encoded_story_valid) if len(e.ids) >= prefix_length + GENERATION_TOKENS][:GENERATION_SIZE]
assert selected, "No validation stories are long enough for generation"
prefixes = torch.tensor([[STORY_BOS] + encoded_story_valid[i].ids[:prefix_length - 1] for i in selected], dtype=torch.long)
prompt_texts = [story_tokenizer.decode(encoded_story_valid[i].ids[:prefix_length - 1]) for i in selected]
reference_texts = [story_tokenizer.decode(encoded_story_valid[i].ids[prefix_length - 1:prefix_length - 1 + GENERATION_TOKENS]) for i in selected]


@torch.inference_mode()
def generate_continuations(model, prefixes, temperature, max_new_tokens):
    model.eval()
    input_ids = prefixes.to(DEVICE)
    logits, state = model(input_ids)
    next_logits = logits[:, -1]
    generated = []
    finished = torch.zeros(len(input_ids), dtype=torch.bool, device=DEVICE)
    entropy_sum = 0.0
    entropy_count = 0

    for _ in range(max_new_tokens):
        next_logits[:, STORY_PAD] = -torch.inf
        next_logits[:, STORY_BOS] = -torch.inf
        next_logits[:, STORY_UNK] = -torch.inf
        probabilities = F.softmax(next_logits / temperature, dim=-1)
        entropy = -(probabilities * probabilities.clamp_min(1e-12).log()).sum(dim=-1)
        active = ~finished
        entropy_sum += entropy[active].sum().item()
        entropy_count += active.sum().item()
        next_ids = torch.multinomial(probabilities, 1).squeeze(1)
        next_ids = torch.where(finished, torch.full_like(next_ids, STORY_EOS), next_ids)
        generated.append(next_ids)
        finished |= next_ids == STORY_EOS
        if finished.all():
            break
        step_logits, state = model(next_ids[:, None], state)
        next_logits = step_logits[:, -1]

    generated = torch.stack(generated, dim=1).cpu().tolist()
    texts = []
    for row in generated:
        clean = []
        for token_id in row:
            if token_id == STORY_EOS:
                break
            if token_id not in {STORY_PAD, STORY_BOS, STORY_UNK}:
                clean.append(token_id)
        texts.append(story_tokenizer.decode(clean))
    return texts, entropy_sum / entropy_count


temperatures = [0.7, 1.0, 1.3]
generations = {}
sampling_entropies = {}
for temperature in temperatures:
    texts, entropy = generate_continuations(lm, prefixes, temperature, GENERATION_TOKENS)
    generations[temperature] = texts
    sampling_entropies[temperature] = entropy

In [34]:
pd.DataFrame({
    "prompt": prompt_texts[:6],
    "reference continuation": reference_texts[:6],
    **{f"T={temperature}": generations[temperature][:6] for temperature in temperatures},
})

,prompt,reference continuation,T=0.7,T=1.0,T=1.3
0,Once upon a time there was a girl named Sally....,and she loved taking pictures with her camera...,"and loved to play with his toys. One day, he ...",". Timmy was a ready to her, Timmy. Her mom cam...",straw when he was too ghost soldier and stron...
1,"Once upon a time, there was a little girl name...",very anxious about touching things. Everywher...,"brave and wanted to explore. One day, she was...",shining and also always wanted to explore. On...,very friendly Go and the party. Jack ro row. ...
2,Once there was a new gorilla. He was walking a...,jungle happily. Then he saw another gorilla t...,"park.\n\nOne day, the bug found a big box. Th...",ado on her stud with hisle. She couldn't take ...,"distance when she pressed his mouth, in his p..."
3,Lily and Ben were best friends. They liked to ...,". One day, they saw a big tent with many peopl...","with her mom.\n\nOne day, Timmy's mom was acr...",. They saw a stormpt. had a story. He fall She...,. he should are playful stronger banana and ev...
4,Lily and Max are playing in the kitchen. They ...,"for Mom. They have flour, eggs, milk, sugar, ...",. Lily liked to explore the park. They liked t...,without he could go on. But we grabbed the ro...,note Ellie app Daisy. Now cheered her ball so...
5,Once there was a little girl who loved to play...,exploring and she found a file in her backyar...,playing in the park. She liked to play with u...,polite and she had a diamond on the forest. H...,scared. O thought she Lila off on yog. Max fo...


Если logits недоступны и остались только тексты, entropy приходится оценивать по частотам сэмплов. Все генерации заново токенизируем одним word-tokenizer, чтобы температуры сравнивались в одинаковых единицах.

Для n-грамм $g$

$$
\widehat p(g)=\frac{c(g)}{N_n},
\qquad
\widehat H_n=-\sum_g\widehat p(g)\log\widehat p(g).
$$

Для контекста $h$ длины $n-1$ conditional n-gram entropy равна

$$
\widehat h_n
=-\sum_{h,w}\frac{c(h,w)}{N_n}
\log\frac{c(h,w)}{c(h)}
=\widehat H(H,W)-\widehat H(H),
$$

где обе entropy считаются на одних и тех же позициях предсказания. Distinct-$n$ измеряет другую величину:

$$
\operatorname{distinct}_n
=\frac{|\{g:c(g)>0\}|}{N_n}.
$$

Это entropy эмпирического корпуса, а не точная sequence entropy модели. Plug-in estimator смещён вниз и зависит от размера выборки.

**Вопрос:** почему exact logit entropy и sample n-gram entropy нельзя отождествлять?

In [ ]:
def generation_statistics(texts):
    tokenized = [WORD_PATTERN.findall(text.lower()) for text in texts]
    result = {"mean_words": np.mean([len(tokens) for tokens in tokenized])}

    for n in [1, 2, 3]:
        counts = Counter(tuple(tokens[i:i + n]) for tokens in tokenized for i in range(len(tokens) - n + 1))
        total = sum(counts.values())
        if total == 0:
            result[f"H{n}"] = np.nan
            result[f"distinct-{n}"] = np.nan
            continue
        probabilities = np.fromiter(counts.values(), dtype=np.float64) / total
        result[f"H{n}"] = -(probabilities * np.log(probabilities)).sum()
        result[f"distinct-{n}"] = len(counts) / total

    return result

metric_rows = []
for temperature in temperatures:
    row = {
        "temperature": temperature,
        "sampled-context entropy": sampling_entropies[temperature],
    }
    row.update(generation_statistics(generations[temperature]))
    metric_rows.append(row)

generation_metrics = pd.DataFrame(metric_rows)
generation_metrics

Entropy измеряет разброс вероятности, но ничего не знает о смысле:

$$
p(v\mid h)=\frac1V
\Rightarrow H=\log V
\quad\text{(максимум разнообразия, возможно полный мусор)},
$$

$$
p(v^*\mid h)\approx1
\Rightarrow H\approx0
\quad\text{(уверенность, возможно бесконечное повторение)}.
$$

## 8. Generative perplexity

Собственная PPL модели плохо подходит для оценки её же сэмплов: разные системы уверены в разных вещах. Поэтому пропустим все продолжения через одну фиксированную внешнюю LM $r_\phi$:

$$
\operatorname{PPL}_{\mathrm{ext}}(S)
=\exp\left(
-\frac{1}{N_r}
\sum_{x\in S}\sum_t
\log r_\phi(x_t\mid x_{<t})
\right).
$$

Это единая fluency-like шкала, но не проверка соответствия prompt, логики сюжета или разнообразия.

H(q) = Int q(x) log q(x) dx
q(x) = 1 / |N|


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

external_tokenizer = AutoTokenizer.from_pretrained("gpt2")
external_tokenizer.pad_token = external_tokenizer.eos_token
external_model = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE).eval()


@torch.inference_mode()
def external_perplexity(texts):
    total_nll = 0.0
    total_tokens = 0
    batch_size = 16 if DEVICE.type == "cuda" else 4
    for start in range(0, len(texts), batch_size):
        batch = [text if text.strip() else "The story ended." for text in texts[start:start + batch_size]]
        encoded = external_tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
        logits = external_model(**encoded).logits[:, :-1]
        targets = encoded["input_ids"][:, 1:]
        mask = encoded["attention_mask"][:, 1:].bool()
        losses = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1), reduction="none").reshape_as(targets)
        total_nll += losses[mask].sum().item()
        total_tokens += mask.sum().item()
    return np.exp(total_nll / total_tokens) if total_tokens else np.nan

external_ppl = {temperature: external_perplexity(generations[temperature]) for temperature in temperatures}
reference_external_ppl = external_perplexity(reference_texts)
external_ppl, reference_external_ppl

In [ ]:
probe_texts = [
    "The little fox found a red ball and happily carried it home.",
    "The the the the the the the the the the.",
    "#####.....###...                ......          ####..",
    "Purple bicycle softly remembers seven hungry clouds yesterday.",
]
pd.DataFrame({"text": probe_texts, "external_PPL": [external_perplexity([text]) for text in probe_texts]})

## 9. MAUVE

External PPL усредняет surprisal, но может не заметить mode collapse. MAUVE сравнивает распределения двух корпусов: тексты переводятся в embeddings, квантуются в $K$ кластеров и образуют дискретные распределения

$$
\widehat p_k=\frac{n_k^{\mathrm{ref}}}{N},
\qquad
\widehat q_k=\frac{n_k^{\mathrm{gen}}}{M}.
$$

Для смесей

$$
r_\lambda=\lambda\widehat p+(1-\lambda)\widehat q
$$

строится преобразованный KL-frontier

$$
\left(
\exp[-cD_{\mathrm{KL}}(\widehat p\Vert r_\lambda)],
\exp[-cD_{\mathrm{KL}}(\widehat q\Vert r_\lambda)]
\right),
\qquad \lambda\in[0,1],
$$

а MAUVE — площадь под этой кривой. На малой выборке embeddings, кластеры и итоговый score шумны.

In [ ]:
# del external_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

import mauve

mauve_scores = {}
for temperature in temperatures:
    output = mauve.compute_mauve(
        p_text=reference_texts,
        q_text=generations[temperature],
        featurize_model_name="gpt2",
        device_id=0 if DEVICE.type == "cuda" else -1,
        max_text_length=128,
        num_buckets=16,
        kmeans_num_redo=1,
        kmeans_max_iter=100,
        batch_size=8,
        verbose=False,
        seed=SEED,
    )
    mauve_scores[temperature] = output.mauve
mauve_scores

## 10. LLM-as-a-Judge

Грамматику, связь с началом истории и непротиворечивость трудно выразить одной автоматической формулой. Попросим instruction-tuned LM применить явный rubric.

Для $K$ критериев абсолютный score агрегируется как

$$
S(\tau)=\frac{1}{NK}
\sum_{i=1}^{N}\sum_{k=1}^{K}
s_{\phi,k}(u_i,y_i^{(\tau)}),
\qquad s_{\phi,k}\in\{1,\ldots,5\}.
$$

В pairwise-режиме

$$
J_\phi(u,a,b)\in\{A,B\}.
$$

Чтобы проверить position bias, дадим ту же пару в обратном порядке:

$$
C_{\mathrm{pos}}
=\frac1N\sum_i
\mathbf 1[\operatorname{id}J(u_i,a_i,b_i)
=\operatorname{id}J(u_i,b_i,a_i)].
$$

Judge остаётся моделью: у него возможны position bias, verbosity bias, self-preference и ошибки следования rubric.

**Вопрос:** сохранится ли выбор после перестановки ответов A и B?

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
judge_tokenizer.padding_side = "left"
if judge_tokenizer.pad_token_id is None:
    judge_tokenizer.pad_token = judge_tokenizer.eos_token
judge_dtype = torch.float16 if DEVICE.type == "cuda" else torch.float32
judge_model = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL, torch_dtype=judge_dtype).to(DEVICE).eval()


@torch.inference_mode()
def ask_judge(prompts, max_new_tokens=32):
    answers = []
    batch_size = 8 if DEVICE.type == "cuda" else 2
    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        chats = [[
            {"role": "system", "content": "You are a strict evaluator. Follow the requested output format exactly."},
            {"role": "user", "content": prompt},
        ] for prompt in batch_prompts]
        rendered = [judge_tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True) for chat in chats]
        encoded = judge_tokenizer(rendered, padding=True, return_tensors="pt").to(DEVICE)
        generated = judge_model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=judge_tokenizer.pad_token_id,
            eos_token_id=judge_tokenizer.eos_token_id,
        )
        prefix_length = encoded["input_ids"].shape[1]
        answers.extend(judge_tokenizer.batch_decode(generated[:, prefix_length:], skip_special_tokens=True))
    return answers

In [ ]:
judge_n = min(JUDGE_SIZE, len(prompt_texts))
score_rows = []

for temperature in temperatures:
    prompts = [
        f"Rate the story continuation from 1 to 5 on four criteria: grammar, coherence with the prompt, consistency, non-repetition. Return only four comma-separated integers in this order.\nPrompt: {prompt_texts[i]}\nContinuation: {generations[temperature][i]}"
        for i in range(judge_n)
    ]
    raw_answers = ask_judge(prompts)
    for answer in raw_answers:
        values = [int(x) for x in re.findall(r"(?<!\d)[1-5](?!\d)", answer)[:4]]
        if len(values) == 4:
            score_rows.append({
                "temperature": temperature,
                "grammar": values[0],
                "coherence": values[1],
                "consistency": values[2],
                "non_repetition": values[3],
            })

judge_scores = pd.DataFrame(score_rows, columns=["temperature", "grammar", "coherence", "consistency", "non_repetition"])
judge_summary = judge_scores.groupby("temperature").mean().reset_index()
judge_summary["judge_mean"] = judge_summary[["grammar", "coherence", "consistency", "non_repetition"]].mean(axis=1)
judge_summary

In [ ]:
low_temperature = temperatures[0]
high_temperature = temperatures[-1]
pair_n = min(PAIR_SIZE, len(prompt_texts))

forward_prompts = [
    f"Choose the better story continuation for grammar, coherence, consistency, and non-repetition. Return only A or B.\nPrompt: {prompt_texts[i]}\nA: {generations[low_temperature][i]}\nB: {generations[high_temperature][i]}"
    for i in range(pair_n)
]
reverse_prompts = [
    f"Choose the better story continuation for grammar, coherence, consistency, and non-repetition. Return only A or B.\nPrompt: {prompt_texts[i]}\nA: {generations[high_temperature][i]}\nB: {generations[low_temperature][i]}"
    for i in range(pair_n)
]

forward_raw = ask_judge(forward_prompts, max_new_tokens=8)
reverse_raw = ask_judge(reverse_prompts, max_new_tokens=8)
forward = [re.search(r"\b[AB]\b", answer.upper()) for answer in forward_raw]
reverse = [re.search(r"\b[AB]\b", answer.upper()) for answer in reverse_raw]
forward = [match.group(0) if match else None for match in forward]
reverse = [match.group(0) if match else None for match in reverse]

forward_identity = ["low" if answer == "A" else "high" if answer == "B" else None for answer in forward]
reverse_identity = ["high" if answer == "A" else "low" if answer == "B" else None for answer in reverse]
valid_pairs = [(a, b, x, y) for a, b, x, y in zip(forward, reverse, forward_identity, reverse_identity) if a and b]

pd.DataFrame([{
    "valid_pairs": len(valid_pairs),
    "position_consistency": np.mean([x == y for _, _, x, y in valid_pairs]) if valid_pairs else np.nan,
    "A_preference_rate": np.mean([a == "A" for a, _, _, _ in valid_pairs] + [b == "A" for _, b, _, _ in valid_pairs]) if valid_pairs else np.nan,
    "low_temperature_win_rate": np.mean([x == "low" for _, _, x, _ in valid_pairs] + [y == "low" for _, _, _, y in valid_pairs]) if valid_pairs else np.nan,
}])

## 11. Benchmarks

* MMLU
* HellaSwag
* Winogrande
* PIQA
* GSM8k
* ...

In [ ]:
benchmark = generation_metrics.copy()
benchmark["external_PPL"] = benchmark["temperature"].map(external_ppl)
benchmark["MAUVE"] = benchmark["temperature"].map(mauve_scores)
benchmark = benchmark.merge(judge_summary[["temperature", "judge_mean"]], on="temperature", how="left")
benchmark.sort_values("temperature")

### Outcome

- TF–IDF is strong when $p(y\mid x)$ is almost determined by lexical tokens.
- RNN sees order, but additional expressiveness doesn't guarantee gains on simple classification.
- The same recurrent backbone produces $C$ logits once for classification and $V$ logits at each step for generation.
- Tokenization defines a tradeoff between $V\leftrightarrow T$, OOV, and the cost of embedding/output head.
- LSTM can be trained from scratch to continue TinyStories, but its generations change with the sampling distribution.
- PPL, logit entropy, sample entropy, external PPL, MAUVE, and LLM-as-a-Judge answer different questions.
- Benchmark is useful as a fixed protocol and metric vector, not as another "master" number.